# Lab W6: Tiga Strategi Representasi Fitur

Terkait: **Bab 01 Section 2.6 · Representasi Fitur: Tiga Pilihan Desain**

## Pertanyaan Lab
Pada dataset yang sama (CIFAR-10), dengan 3 kondisi ukuran dataset, strategi representasi mana yang paling menguntungkan?

## Tiga Strategi yang Dibandingkan
| Strategi | Deskripsi | Pretrained? | Yang Dilatih |
|---|---|---|---|
| **Engineered** | HOG features + MLP kecil | Tidak | MLP seluruhnya dari nol |
| **Extracted** | ResNet-18 frozen → linear probe | Ya (ImageNet) | Hanya head linear |
| **Learned** | ResNet-18 fine-tune penuh | Ya (ImageNet) | Seluruh jaringan |

## Tujuan
1. Implementasikan ketiga strategi dan bandingkan pada dataset penuh dan dataset kecil.
2. Pahami mengapa perbedaannya terjadi, bukan hanya angkanya.
3. Identifikasi kapan setiap strategi paling menguntungkan.

## Checklist
- [ ] Ketiga strategi menghasilkan hasil yang dapat dibandingkan (input/output shape sama, metrik sama).
- [ ] Eksperimen pada dataset penuh (50k) DAN dataset kecil (500 sampel per kelas = 5k).
- [ ] Tabel perbandingan dan plot bar.
- [ ] Jawaban refleksi menggunakan pengamatan konkret dari hasil.
Terkait: **Bab 01 Section 2.6 · Representasi Fitur: Tiga Pilihan Desain**

## Pertanyaan Lab
Pada dataset yang sama (CIFAR-10), dengan 3 kondisi ukuran dataset, strategi representasi mana yang paling menguntungkan?

## Tiga Strategi yang Dibandingkan
| Strategi | Deskripsi | Pretrained? | Yang Dilatih |
|---|---|---|---|
| **Engineered** | HOG features + MLP kecil | Tidak | MLP seluruhnya dari nol |
| **Extracted** | ResNet-18 frozen → linear probe | Ya (ImageNet) | Hanya head linear |
| **Learned** | ResNet-18 fine-tune penuh | Ya (ImageNet) | Seluruh jaringan |

## Tujuan
1. Implementasikan ketiga strategi dan bandingkan pada dataset penuh dan dataset kecil.
2. Pahami mengapa perbedaannya terjadi, bukan hanya angkanya.
3. Identifikasi kapan setiap strategi paling menguntungkan.

## Checklist
- [ ] Ketiga strategi menghasilkan hasil yang dapat dibandingkan (input/output shape sama, metrik sama).
- [ ] Eksperimen pada dataset penuh (50k) DAN dataset kecil (500 sampel per kelas = 5k).
- [ ] Tabel perbandingan dan plot bar.
- [ ] Jawaban refleksi menggunakan pengamatan konkret dari hasil.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
print('Device:', DEVICE)
print('torch:', torch.__version__)

## 1. Data: penuh dan subset kecil

In [ ]:
# Transform untuk CNN/pretrained (224x224 karena ResNet)
transform_resnet = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # ImageNet stats
])

# Transform untuk HOG (bisa ukuran lebih kecil)
transform_hog = T.Compose([
    T.Resize(32),
    T.ToTensor(),
])

data_root = '../data'

# Dataset penuh
cifar_train_full = torchvision.datasets.CIFAR10(data_root, train=True, download=True, transform=transform_resnet)
cifar_test       = torchvision.datasets.CIFAR10(data_root, train=False, download=False, transform=transform_resnet)

# Dataset kecil: 500 sampel per kelas (5000 total)
def make_balanced_subset(dataset, samples_per_class: int, n_classes: int = 10):
    indices = {c: [] for c in range(n_classes)}
    for idx, (_, label) in enumerate(dataset):
        if len(indices[label]) < samples_per_class:
            indices[label].append(idx)
        if all(len(v) == samples_per_class for v in indices.values()):
            break
    flat = [idx for idxs in indices.values() for idx in idxs]
    return Subset(dataset, flat)

cifar_train_small = make_balanced_subset(cifar_train_full, samples_per_class=500)

print(f'Train full:  {len(cifar_train_full)} sampel')
print(f'Train small: {len(cifar_train_small)} sampel (500/kelas)')
print(f'Test:        {len(cifar_test)} sampel')

## 2. Strategi A: Engineered (HOG + MLP)

HOG (Histogram of Oriented Gradients) adalah fitur klasik yang menangkap tepi dan gradien di berbagai orientasi. Dirancang oleh manusia berdasarkan pengetahuan tentang apa yang membedakan objek.

In [ ]:
try:
    from skimage.feature import hog
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'scikit-image', '-q'])
    from skimage.feature import hog

def extract_hog_features(dataset, batch_size=512):
    """Ekstrak HOG features dari dataset (tidak perlu GPU)."""
    # Gunakan transform 32x32 untuk HOG (bukan 224)
    hog_transform = T.Compose([T.Resize(32), T.ToTensor()])

    all_features, all_labels = [], []
    # Akses dataset asli (bukan transform resnet)
    raw_dataset = dataset.dataset if isinstance(dataset, Subset) else dataset
    indices = dataset.indices if isinstance(dataset, Subset) else range(len(dataset))

    for idx in indices:
        img_pil, label = raw_dataset.data[idx], raw_dataset.targets[idx]
        from PIL import Image
        img_pil = Image.fromarray(img_pil)
        img_t = hog_transform(img_pil).numpy()  # (3, 32, 32)
        img_gray = img_t.mean(axis=0)  # (32, 32) - HOG pada grayscale
        feat = hog(img_gray, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), visualize=False)
        all_features.append(feat)
        all_labels.append(label)

    return np.array(all_features), np.array(all_labels)

print('Mengekstrak HOG dari train small...')
t0 = time.time()
X_hog_train, y_hog_train = extract_hog_features(cifar_train_small)
X_hog_test, y_hog_test   = extract_hog_features(cifar_test)
print(f'Selesai ({time.time()-t0:.1f}s). HOG feature dim: {X_hog_train.shape[1]}')

In [ ]:
# Normalisasi fitur (wajib untuk MLP)
scaler = StandardScaler()
X_hog_train_norm = scaler.fit_transform(X_hog_train)  # fit pada train only!
X_hog_test_norm  = scaler.transform(X_hog_test)        # transform test dengan scaler yang sama

hog_dim = X_hog_train.shape[1]

class MLP(nn.Module):
    def __init__(self, in_dim, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256),    nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.net(x)

def train_mlp(X_train, y_train, X_test, y_test, epochs=50, lr=1e-3):
    Xt = torch.tensor(X_train, dtype=torch.float32)
    yt = torch.tensor(y_train, dtype=torch.long)
    Xv = torch.tensor(X_test, dtype=torch.float32)
    yv = torch.tensor(y_test, dtype=torch.long)

    loader = DataLoader(list(zip(Xt, yt)), batch_size=256, shuffle=True)
    model = MLP(Xt.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            criterion(model(xb), yb).backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        acc = (model(Xv.to(DEVICE)).argmax(1).cpu() == yv).float().mean().item()
    return acc

print('Training HOG+MLP...')
t0 = time.time()
acc_hog = train_mlp(X_hog_train_norm, y_hog_train, X_hog_test_norm, y_hog_test)
print(f'HOG+MLP test acc: {acc_hog*100:.1f}% ({time.time()-t0:.1f}s)')

## 3. Strategi B: Extracted (ResNet-18 frozen → linear probe)

Ambil *hidden states* dari ResNet-18 yang sudah dilatih pada ImageNet. Bekukan seluruh jaringan - tidak ada parameter yang dilatih ulang. Hanya tambahkan layer linear di atasnya.

In [ ]:
def extract_resnet_features(dataset, batch_size=128):
    """Ekstrak hidden states dari ResNet-18 yang dibekukan."""
    backbone = torchvision.models.resnet18(weights='IMAGENET1K_V1')
    # Hapus classifier head, ambil output dari avg pool
    backbone.fc = nn.Identity()
    backbone.eval().to(DEVICE)
    for p in backbone.parameters():
        p.requires_grad = False  # Bekukan semua parameter

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    all_features, all_labels = [], []

    with torch.no_grad():
        for x, y in loader:
            feats = backbone(x.to(DEVICE)).cpu().numpy()
            all_features.append(feats)
            all_labels.append(torch.as_tensor(y).numpy())

    return np.vstack(all_features), np.concatenate(all_labels)

print('Mengekstrak ResNet features (frozen)...')
t0 = time.time()
X_resnet_train_small, y_rsnet_train = extract_resnet_features(cifar_train_small)
X_resnet_test, y_resnet_test         = extract_resnet_features(cifar_test)
print(f'Selesai ({time.time()-t0:.1f}s). ResNet feature dim: {X_resnet_train_small.shape[1]}')

In [ ]:
# Linear probe: hanya latih layer linear di atas fitur yang diekstrak
scaler_r = StandardScaler()
X_r_train_norm = scaler_r.fit_transform(X_resnet_train_small)
X_r_test_norm  = scaler_r.transform(X_resnet_test)

print('Training linear probe...')
t0 = time.time()
acc_extracted = train_mlp(X_r_train_norm, y_rsnet_train, X_r_test_norm, y_resnet_test,
                           epochs=100, lr=1e-3)
print(f'Extracted (frozen ResNet + linear probe) test acc: {acc_extracted*100:.1f}% ({time.time()-t0:.1f}s)')

## 4. Strategi C: Learned (ResNet-18 fine-tune penuh)

Mulai dari ResNet-18 pretrained, tapi semua parameter ikut dilatih. Model *belajar representasi yang spesifik untuk CIFAR-10*, bukan hanya menggunakan representasi ImageNet.

In [ ]:
def finetune_resnet(train_dataset, test_dataset, epochs=15, lr=1e-4, freeze=False):
    model = torchvision.models.resnet18(weights='IMAGENET1K_V1')
    model.fc = nn.Linear(512, 10)  # Ganti head untuk 10 kelas CIFAR-10

    if freeze:
        # Frozen: hanya fc yang dilatih (ini = extracted + linear)
        for name, p in model.named_parameters():
            if 'fc' not in name:
                p.requires_grad = False
    model = model.to(DEVICE)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x = x.to(DEVICE)
            y = torch.as_tensor(y).long().view(-1).to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            criterion(model(x), y).backward()
            optimizer.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            y = torch.as_tensor(y).long().view(-1).to(DEVICE)
            correct += (model(x).argmax(1) == y).sum().item()
            total += len(y)
    return correct / total

print('Fine-tuning ResNet-18 (dataset kecil, 15 epoch)...')
t0 = time.time()
acc_learned_small = finetune_resnet(cifar_train_small, cifar_test, epochs=15, lr=1e-4)
print(f'Learned (fine-tune penuh, small data) test acc: {acc_learned_small*100:.1f}% ({time.time()-t0:.1f}s)')

## 5. Perbandingan pada dataset penuh (opsional, butuh lebih lama)

In [ ]:
RUN_FULL = False  # Ganti ke True untuk jalankan eksperimen dataset penuh (~30-60 menit tanpa GPU)

if RUN_FULL:
    print('Mengekstrak ResNet features (full dataset)...')
    X_resnet_train_full, y_r_train_full = extract_resnet_features(cifar_train_full)
    scaler_rf = StandardScaler()
    X_rf_train_norm = scaler_rf.fit_transform(X_resnet_train_full)
    X_rf_test_norm  = scaler_rf.transform(X_resnet_test)

    acc_extracted_full = train_mlp(X_rf_train_norm, y_r_train_full, X_rf_test_norm, y_resnet_test,
                                    epochs=100)
    print(f'Extracted (full data): {acc_extracted_full*100:.1f}%')

    print('Fine-tuning ResNet (full dataset)...')
    acc_learned_full = finetune_resnet(cifar_train_full, cifar_test, epochs=30, lr=1e-4)
    print(f'Learned (full data): {acc_learned_full*100:.1f}%')
else:
    acc_extracted_full = None
    acc_learned_full = None
    print('RUN_FULL=False. Lanjutkan dengan hasil dataset kecil saja.')

## 6. Tabel perbandingan dan visualisasi

In [ ]:
import pandas as pd

results = {
    'Strategi':  ['Engineered\n(HOG+MLP)',
                  'Extracted\n(ResNet frozen)',
                  'Learned\n(ResNet fine-tune)'],
    'Dataset Kecil (5k)': [acc_hog * 100, acc_extracted * 100, acc_learned_small * 100],
    'Dataset Penuh (50k)': [None, acc_extracted_full * 100 if acc_extracted_full else None,
                            acc_learned_full * 100 if acc_learned_full else None],
}
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(3)
small_vals = [acc_hog * 100, acc_extracted * 100, acc_learned_small * 100]
colors = ['#4C8BB5', '#5BAD6F', '#E07B54']
bars = ax.bar(x, small_vals, color=colors, alpha=0.85, edgecolor='white', width=0.5)

for bar, val in zip(bars, small_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5, f'{val:.1f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(['Engineered\n(HOG+MLP)', 'Extracted\n(ResNet frozen)', 'Learned\n(ResNet fine-tune)'],
                    fontsize=10)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Perbandingan Strategi Representasi\nCIFAR-10, Dataset Kecil (500 sampel/kelas)')
ax.set_ylim(0, max(small_vals) + 10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
Path('../experiments/plots').mkdir(parents=True, exist_ok=True)
plt.savefig('../experiments/plots/lab1b_representasi.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Refleksi

1. **Perbedaan terbesar** ada di strategi mana? Apakah perbedaan ini masuk akal berdasarkan mekanisme yang kamu pahami?

2. **Extracted vs Learned**: Pada dataset kecil (5k), strategi mana yang lebih menguntungkan? Mengapa *extracted* bisa lebih baik dari *learned* ketika data sedikit? (Petunjuk: pikirkan tentang overfitting dan distribusi ImageNet vs CIFAR-10)

3. **Trade-off waktu**: Catat waktu training masing-masing strategi dari output sel di atas. Jika dataset yang kamu kerjakan di lab riset hanya punya 200 sampel per kelas dan kamu punya deadline 3 hari, strategi mana yang dipilih dan mengapa?

4. **HOG vs Learned**: Pada angka yang kamu dapat, seberapa jauh *engineered* tertinggal dari *learned*? Apakah selisih itu cukup besar untuk membenarkan biaya fine-tuning dalam konteks riset dengan data sangat terbatas?

5. **Koneksi ke Section 2.6**: Pilih satu keputusan dari tabel keputusan di Section 2.6 (frozen vs fine-tuned, layer mana, pooling apa) dan jelaskan bagaimana lab ini memberikan bukti konkret untuk atau melawan keputusan itu.

### Jawaban Refleksi

**1. Perbedaan terbesar dan penjelasannya:**
> *[tulis di sini]*

**2. Extracted vs Learned pada data kecil:**
> *[tulis di sini]*

**3. Trade-off waktu:**
> *[tulis di sini]*

**4. HOG vs Learned - apakah selisihnya worth it:**
> *[tulis di sini]*

**5. Koneksi ke Section 2.6:**
> *[tulis di sini]*